In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py

In [ ]:
db = Db("~/.clover/p801.cfg")
query = """
    SELECT
    developer.id AS 'd.id',
    developer_app.id AS 'da.id',
    developer.created_time AS 'd.created_time',
    developer_app.first_approval_time AS 'da.first_approval_time',
    TIMESTAMPDIFF(DAY, developer.created_time, developer_app.first_approval_time) AS 'days_to_approval'
    FROM developer_app
    JOIN developer on developer.id = developer_app.developer_id
    JOIN account on account.id = developer.owner_account_id
    WHERE developer.created_time > '0000-00-00'
    AND developer_app.first_approval_time > '0000-00-00'
    AND""" + NOT_LIKE_DEVELOPER_NAMES_ACCOUNT_EMAILS + """
    AND account.last_login IS NOT NULL
    """
df = pd.read_sql(query, con=db.conn)
db.close()

# Create a new dataframe with each developer's first approved app.
df = (df
      .sort_values(by=['da.first_approval_time'])
      .groupby('d.id')
      .first()
      .reset_index())

df

In [ ]:
print df.describe()
print "Range of days to approval: " + str(df['days_to_approval'].min()) + ", " + str(df['days_to_approval'].max())
print "Mean (average) days to approval: " + str(df['days_to_approval'].mean())
print "Median days to approval: " + str(df['days_to_approval'].median())
print "Modal days to approval: " + str(df['days_to_approval'].mode().values[0])